<a href="https://colab.research.google.com/github/ivanriuk/Tarea4_Optimizacion2/blob/main/ExactMDVRP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import Pkg; Pkg.add("HiGHS")
import Pkg; Pkg.add("JuMP")

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed CodecBzip2 ───────── v0.8.5
   Installed OpenBLAS32_jll ───── v0.3.30+0
   Installed HiGHS_jll ────────── v1.14.0+0
   Installed MathOptIIS ───────── v0.2.0
   Installed HiGHS ────────────── v1.23.0
   Installed BenchmarkTools ───── v1.7.0
   Installed MutableArithmetics ─ v1.7.1
   Installed MathOptInterface ─── v1.50.1
    Updating `~/.julia/environments/v1.11/Project.toml`
  [87dc4568] + HiGHS v1.23.0
    Updating `~/.julia/environments/v1.11/Manifest.toml`
  [6e4b80f9] + BenchmarkTools v1.7.0
  [523fee87] + CodecBzip2 v0.8.5
  [87dc4568] + HiGHS v1.23.0
  [8c4f8055] + MathOptIIS v0.2.0
  [b8f27783] + MathOptInterface v1.50.1
  [d8a4904e] + MutableArithmetics v1.7.1
  [8fd58aa0] + HiGHS_jll v1.14.0+0
  [656ef2d0] + OpenBLAS32_jll v0.3.30+0
  [9abbd945] + Profile v1.11.0
Precompiling project...
   4555.3 ms  ✓ HiGHS_jll
   4563.4 ms  ✓ OpenBLAS32_jll
   1749.5 ms  ✓ CodecBzip2
  

In [7]:
using JuMP
using HiGHS

"""
Construye un modelo MDVRP basado en la formulación corregida del paper.

Entradas
--------
c           :: Matrix{Float64}   # matriz de costos/distancias de tamaño (m+n) x (m+n)
demand      :: Vector{Float64}   # demandas de los clientes, longitud n
depot_cap   :: Vector{Float64}   # capacidad de cada depósito, longitud m
veh_cap     :: Vector{Float64}   # capacidad de cada vehículo, longitud K
route_cap   :: Vector{Float64}   # longitud máxima de cada ruta, longitud K

"""

function build_mdvrp_model(
    c::Matrix{Float64},
    demand::Vector{Float64},
    depot_cap::Vector{Float64},
    veh_cap::Vector{Float64},
    route_cap::Vector{Float64}
)

    m = length(depot_cap)              # número de depósitos
    n = length(demand)                 # número de clientes
    Knum = length(veh_cap)             # número de vehículos

    @assert size(c, 1) == m + n
    @assert size(c, 2) == m + n
    @assert length(route_cap) == Knum

    # Conjuntos
    I = 1:m                            # depósitos
    J = (m + 1):(m + n)                # clientes
    V = 1:(m + n)                      # todos los nodos
    K = 1:Knum                         # vehículos

    # Vector de demanda definido sobre todos los nodos
    dem = zeros(Float64, m + n)
    dem[J] = demand

    model = Model(HiGHS.Optimizer)

    # =========================
    # Variables
    # =========================
    @variable(model, x[V, V, K], Bin)              # x[i,j,k] = 1 si k usa el arco i->j
    @variable(model, U[J, K] >= 0)            # variable auxiliar U

    # =========================
    # Función objetivo
    # =========================
    @objective(model, Min,
        sum(c[i, j] * x[i, j, k] for i in V, j in V, k in K))

    # =========================
    # Restricciones auxiliares útiles
    # =========================

    # No self-loops
    @constraint(model, [v in V, k in K], x[v, v, k] == 0)

    # No arcos depósito -> depósito
    @constraint(model, [i in I, h in I, k in K; i != h], x[i, h, k] == 0)

    # =========================
    # (2) Cada cliente debe ser visitado exactamente una vez
    # =========================
    @constraint(model, [j in J],
        sum(x[i, j, k] for i in V, k in K) == 1
    )

    # =========================
    # (3) Capacidad del vehículo
    # =========================
    @constraint(model, [k in K],
        sum(
            dem[j] * sum(x[i, j, k] for i in V)
            for j in J
        ) <= veh_cap[k]
    )

    # =========================
    # (4) Eliminación de subtours (MTZ)
    # U[l,k] - U[j,k] + n*x[l,j,k] <= n-1
    # =========================
    @constraint(model, [k in K, l in J, j in J; l != j],
        U[l, k] - U[j, k] + n * x[l, j, k] <= n - 1
    )

    # Estas dos restricciones ayudan a que U[j,k] solo "se active"
    # si el cliente j realmente pertenece a la ruta k
    @constraint(model, [j in J, k in K],
        U[j, k] <= n * sum(x[i, j, k] for i in V)
    )

    @constraint(model, [j in J, k in K],
        U[j, k] >= sum(x[i, j, k] for i in V)
    )

    # =========================
    # (5) Conservación de flujo
    # =========================
    @constraint(model, [v in V, k in K],
        sum(x[v, j, k] for j in V) == sum(x[j, v, k] for j in V)
    )

    # =========================
    # (6) Cada vehículo sale a lo más una vez de algún depósito
    # =========================
    @constraint(model, [k in K],
        sum(x[i, j, k] for i in I, j in J) <= 1
    )

    # =========================
    # (9) Longitud máxima de la ruta
    # =========================
    @constraint(model, [k in K],
        sum(c[i, j] * x[i, j, k] for i in V, j in V) <= route_cap[k]
    )

    return model, x, U
end

build_mdvrp_model (generic function with 1 method)

In [8]:
m = 2
n = 5
K = 3

# Matriz de costos/distancias entre todos los nodos: depósitos + clientes
c = [
    0.0  4.0  6.0  8.0  7.0  3.0  5.0;
    4.0  0.0  5.0  6.0  4.0  7.0  3.0;
    6.0  5.0  0.0  2.0  3.0  6.0  4.0;
    8.0  6.0  2.0  0.0  4.0  5.0  3.0;
    7.0  4.0  3.0  4.0  0.0  2.0  6.0;
    3.0  7.0  6.0  5.0  2.0  0.0  4.0;
    5.0  3.0  4.0  3.0  6.0  4.0  0.0
]

demand    = [2.0, 3.0, 4.0, 2.0, 1.0]      # clientes
depot_cap = [8.0, 10.0]                    # depósitos
veh_cap   = [5.0, 5.0, 5.0]                # vehículos
route_cap = [20.0, 20.0, 20.0]             # máximo recorrido por vehículo

model, x, U= build_mdvrp_model(
    c, demand, depot_cap, veh_cap, route_cap
)

optimize!(model)

println("Estado: ", termination_status(model))
println("Objetivo: ", objective_value(model))

Running HiGHS 1.14.0 (git hash: 7df0786de3): Copyright (c) 2026 under Apache 2.0 license terms
Using BLAS: blastrampoline 
MIP has 189 rows; 180 cols; 1453 nonzeros; 157 integer variables (157 binary)
Coefficient ranges:
  Matrix  [1e+00, 8e+00]
  Cost    [2e+00, 1e+03]
  Bound   [1e+00, 5e+00]
  RHS     [1e+00, 2e+01]
Presolving model
157 rows, 148 cols, 1308 nonzeros 0s
157 rows, 148 cols, 1284 nonzeros 0s
Presolve reductions: rows 157(-32); columns 148(-32); nonzeros 1284(-169) 
Objective function is integral with scale 1

Solving MIP model with:
   157 rows
   148 cols (125 binary, 0 integer, 8 implied int., 15 continuous, 0 domain fixed)
   1284 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial

In [9]:
m = length(depot_cap)
n = length(demand)
Knum = length(veh_cap)
I = 1:m
J = (m+1):(m+n)
V = 1:(m+n)
Kset = 1:Knum

println("\nArcos usados:")
for k in Kset, i in V, j in V
    if value(x[i, j, k]) > 0.5
        println("vehículo $k: $i -> $j")
    end
end

println("\nValores U (orden MTZ):")
for k in Kset, j in J
    if value(U[j, k]) > 1e-6
        println("U[$j,$k] = ", value(U[j, k]))
    end
end


Arcos usados:
vehículo 1: 2 -> 3
vehículo 1: 3 -> 4
vehículo 1: 4 -> 2
vehículo 2: 1 -> 6
vehículo 2: 6 -> 1
vehículo 3: 2 -> 7
vehículo 3: 5 -> 2
vehículo 3: 7 -> 5

Asignación cliente-depósito:
cliente 6 asignado al depósito 1
cliente 3 asignado al depósito 2
cliente 4 asignado al depósito 2
cliente 5 asignado al depósito 2
cliente 7 asignado al depósito 2

Valores U (orden MTZ):
U[3,1] = 1.000000000000022
U[4,1] = 1.9999999999999947
U[6,2] = 0.9999999999999621
U[5,3] = 2.00000000000001
U[7,3] = 0.9999999999999973
